# Material de aula - FACENS - Aprendizado de Máquina

# <center> RESOLUÇÃO DA ATIVIDADE </center>


## ADALINE e Regra Delta - Laboratório 02

### Utilize o algoritmo de treinamento da Regra Delta para classificação de padrões no ADALINE, com os seguintes parâmetros: 

### 1. Taxa de aprendizado $ \eta = 0,0025 $
### 2. Precisão $ \epsilon = 10^{-6} $

Realize as seguintes atividades:

In [1]:
#CÉLULA ADALINE-LIB-01
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
#from enum import Enum
#from operator import xor
%matplotlib inline

In [2]:
#CÉLULA ADALINE-LIB-02
#Função para carregamento de dados de treinamento
def loadDataFile(filename) :
    '''
    Esta função faz o carregamento de dados de um arquivo separado por virgulas (csv)
    
    Parametros
    -----------
    filename -> nome do arquivo de dados em formato csv
    
    
    Retorno
    -----------  
    dataframe contendo os dados do arquivo formatados numericamente.
    '''    
    dfFile = pd.read_csv(filename, sep=';', decimal=',', thousands='.')
    
    return dfFile

# 1 - Criando a função de escala - Parte 1

Utilizando Jupyter e NumPy: para a função de variável contínua apresentada, construa uma função para o cálculo da escala dos dados para o domínio da função de ativação.

Assinatura: **def tales(x1, x1_min, x1_max, x2_min, x2_max) :**

### esta função faz o escalamento de 1 amostra x1, que pertence ao intervalo [x1_min, x1_max],
### dentro do intervalo [x2_min, x2_max]
    
**Retorno da Função:** valor dentro do domínio dado

In [3]:
#CÉLULA ADALINE-LIB-03
# Função para cálculo de normalização de uma amostra
def tales(x1, x1_min, x1_max, x2_min, x2_max) :
    '''
    Esta função faz o escalamento de 1 amostra x1, que pertence ao intervalo [x1_min, x1_max], 
    dentro do intervalo [x2_min, x2_max]
    
    Parametros
    -----------
    x1     -> valor a ser normalizado
    x1_min -> valor minimo atingido por x1
    x1_max -> valor maximo atingido por x1
    x2_min -> valor minimo da escala de destino
    x2_max -> valor maximo da escala de destino    
    
    Retorno
    -----------  
    valor de x1, que pertence ao intervalo [x1_min, x1_max], projetado dentro do intervalo [x2_min, x2_max]
    '''
    
    #RESOLUÇÃO - INICIO

    #Teorema de Tales
    # ( x2 - x2_min ) / ( x2_max - x2_min ) = ( x1 - x1_min ) / (x1_max - % x1_min);
    
    #Definindo x1 como a escala do vetor dado e x2 como a escala de saída (variavel de interesse), isola-se x2:
    x2 = ( ( x1 - x1_min ) / (x1_max - x1_min) )* ( x2_max - x2_min ) + x2_min

    #RESOLUÇÃO - FIM

    return x2

In [4]:
#Teste
print(tales(5, -10, 10, -1, 1)) #Valor esperado: 0.5

print(tales(-5, -10, 10, -1, 1)) #Valor esperado: -0.5

print(tales(-10, -10, 10, -1, 1)) #Valor esperado: -1.0

print(tales(10, -10, 10, -1, 1)) #Valor esperado: 1.0

0.5
-0.5
-1.0
1.0


## Validação
#Teste
print(tales(5, -10, 10, -1, 1)) #Valor esperado: 0.5

print(tales(-5, -10, 10, -1, 1)) #Valor esperado: -0.5

print(tales(-10, -10, 10, -1, 1)) #Valor esperado: -1.0

print(tales(10, -10, 10, -1, 1)) #Valor esperado: 1.0

# 2 - Criando a função de escala - Parte 2

Utilizando Jupyter e NumPy: Utilizando Jupyter e NumPy: para a função de variável contínua apresentada, construa uma função para o cálculo da escala do vetor de dados para o domínio da função de ativação.

Assinatura: **def scale(x2_min, x2_max, sampleArray) :**

### esta função faz o escalamento de 1 array de dados que para dentro do intervalo [x2_min, x2_max]
    
**Retorno da Função:** valor dentro do domínio dado

**NOTA**: Não sabemos os limites da escala do array, temos que descobrir!

In [5]:
#CÉLULA ADALINE-LIB-04
def scale(x2_min, x2_max, sampleArray) :    
    '''
    Esta função faz o escalamento de 1 vetor de amostras para dentro do intervalo [x2_min, x2_max]
    Ela retorna um vetor com os dados normalizados dentro do dominio dado
    
    Parametros
    -----------
    x2_min      -> valor minimo da escala de destino
    x2_max      -> valor maximo da escala de destino
    sampleArray -> vetor a ser normalizado   
    
    Retorno
    -----------  
    vetor sampleArray projetado dentro do intervalo [x2_min, x2_max]   
    '''
    
    #RESOLUÇÃO - INICIO
    x_max = max(sampleArray)
    x_min = min(sampleArray)
    
    sampleArrayNorm = [tales(x, x_min, x_max, x2_min, x2_max) for x in sampleArray]
    #RESOLUÇÃO - FIM
    
    return sampleArrayNorm

In [6]:
#Teste
v = np.array([5, -5, 10, -10, 0, 3, -3, 4, -4, 7])
scale(-1, 1, v)

[0.5,
 -0.5,
 1.0,
 -1.0,
 0.0,
 0.30000000000000004,
 -0.30000000000000004,
 0.3999999999999999,
 -0.4,
 0.7]

## Validação
``
v = np.array([5, -5, 10, -10, 0, 3, -3, 4, -4, 7])
scale(-1, 1, v)
``

**Valor esperado:**
    [0.5, -0.5, 1.0, -1.0, 0.0, 0.3, -0.3, 0.39, -0.4, 0.7]

# 3 - Teste sua função de escala com os dados do problema

Faça a carga dos dados e verifique se consegue fazer o escalamento das colunas de dados de treinamento e operação

In [7]:
#Carregando dataset de treino
dfTrainingFile = loadDataFile('Lab01_training.csv')
dfTrainingFile

,Amostra,x1,x2,x3,x4,d
0,1,0.4329,-1.3719,0.7022,-0.8535,1
1,2,0.3024,0.2286,0.8630,2.7909,-1
2,3,0.1349,-0.6445,1.0530,0.5687,-1
3,4,0.3374,-1.7163,0.3670,-0.6283,-1
4,5,1.1434,-0.0485,0.6637,1.2606,1
5,6,1.3749,-0.5071,0.4464,1.3009,1
6,7,0.7221,-0.7587,0.7681,-0.5592,1
7,8,0.4403,-0.8072,0.5154,-0.3129,1
8,9,-0.5231,0.3548,0.2538,1.5776,-1
9,10,0.3255,-2.0000,0.7112,-1.1209,1


In [8]:
#Normalizando dados de treino
dfTrainingFileNorm = dfTrainingFile.copy()
dfTrainingFileNorm['x1'] = scale(-1,1, dfTrainingFile['x1'])
dfTrainingFileNorm['x2'] = scale(-1,1, dfTrainingFile['x2'])
dfTrainingFileNorm['x3'] = scale(-1,1, dfTrainingFile['x3'])

dfTrainingFileNorm

,Amostra,x1,x2,x3,x4,d
0,1,-0.019839,-0.308281,0.452773,-0.8535,1
1,2,-0.153637,0.286612,0.703611,2.7909,-1
2,3,-0.325370,-0.037913,1.000000,0.5687,-1
3,4,-0.117753,-0.436292,-0.070119,-0.6283,-1
4,5,0.708617,0.183616,0.392715,1.2606,1
5,6,0.945968,0.013158,0.053740,1.3009,1
6,7,0.276670,-0.080360,0.555573,-0.5592,1
7,8,-0.012252,-0.098387,0.161376,-0.3129,1
8,9,-1.000000,0.333519,-0.246705,1.5776,-1
9,10,-0.129953,-0.541741,0.466812,-1.1209,1


In [9]:
#Carregando dataset de teste
dfTestFile = loadDataFile('Lab01_operation.csv')
dfTestFile

,Amostra,x1,x2,x3,x4
0,1,0.9694,0.6909,0.4334,3.4965
1,2,0.5427,1.3832,0.6390,4.0352
2,3,0.6081,-0.9196,0.5925,0.1016
3,4,-0.1618,0.4694,0.2030,3.0117
4,5,0.1870,-0.2578,0.6124,1.7749
5,6,0.4891,-0.5276,0.4378,0.6439
6,7,0.3777,2.0149,0.7423,3.3932
7,8,1.1498,-0.4067,0.2469,1.5866
8,9,0.9325,1.0950,1.0359,3.3591
9,10,0.5060,1.3317,0.9222,3.7174


In [10]:
#Normalizando dados de teste
dfTestFileNorm = dfTestFile.copy()
dfTestFileNorm['x1'] = scale(-1,1, dfTestFileNorm['x1'])
dfTestFileNorm['x2'] = scale(-1,1, dfTestFileNorm['x2'])
dfTestFileNorm['x3'] = scale(-1,1, dfTestFileNorm['x3'])

dfTestFileNorm

,Amostra,x1,x2,x3,x4
0,1,0.267795,-0.015975,-0.446752,3.4965
1,2,-0.199847,0.231165,0.046944,4.0352
2,3,-0.128171,-0.590897,-0.064714,0.1016
3,4,-0.971944,-0.095047,-1.000000,3.0117
4,5,-0.589676,-0.354645,-0.016929,1.7749
5,6,-0.258590,-0.450959,-0.436187,0.6439
6,7,-0.380678,0.456671,0.294993,3.3932
7,8,0.465505,-0.407800,-0.894585,1.5866
8,9,0.227355,0.128282,1.000000,3.3591
9,10,-0.240068,0.212780,0.726978,3.7174
